# **Lab 7: Nearest Neighbours**

Classification means predicting a **category** rather than a number.

The idea behind nearest neighbours is this: to decide what a new point is, find the
points most like it and see what they are.

"Most like it" needs a definition, and that is what this lab builds. By the end
you will have measured the distance between two shots, used it to classify a
shot you have never seen, and checked how often that classification is right.
The distance function and the train/test split are used again in Project 2,
which goes further: it compares against several nearby points rather than one.

#### **Helpful Resource:**
- [Python Reference](https://ulwazi.wits.ac.za/courses/89081/pages/detailed-python-reference-sheet-python-cheat-sheet-2?module_item_id=1200407)

**Recommended Readings:**
- [17.1 Nearest Neighbors](https://inferentialthinking.com/chapters/17/1/Nearest_Neighbors.html),
- [17.2 Training and Testing](https://inferentialthinking.com/chapters/17/2/Training_and_Testing)
and
- [17.3 Rows of Tables](https://inferentialthinking.com/chapters/17/3/Rows_of_Tables.html).

There are no hidden tests. Every test you can run is every test that counts.

## **Do not change the code in the two cells below** ##

In [ ]:
# Run this cell first to install the libraries. It may take a minute.
%pip install -q urllib3<2.0 otter-grader==7.0.0 datascience ipywidgets

In [ ]:
# Run this cell to set up the notebook, but please do not change it.
try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

import numpy as np
from datascience import *
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

import otter
grader = otter.Notebook("lab7.ipynb")

---
---

### **If something goes wrong** (Refer to this if get an error)

Errors are normal. Everyone gets them, including people who have been doing this
for years. The message tells you what happened, once you know how to read it.

| What you see | What it usually means |
|---|---|
| `NameError: name 'x' is not defined` | You have not run the cell that creates `x`, or you typed it differently |
| `NameError: name 'Table' is not defined` | The setup cell at the very top has not been run. That is the cell that brings in `Table`, `np` and the plotting tools. Run it, then run your own cell again |
| `NameError: name 'grader' is not defined` | The same cause, for the cell that sets up the tests. If you did run it, read its output: the `otter` install needs a connection the first time, and if it failed then every `grader.check` below it fails too. Run that cell again |
| `ValueError: label "X" not in labels` | That column does not exist. Run `tbl.labels` to see the real names |
| `SyntaxError` | A missing bracket or quote, often on the line **above** the one it points at |
| `TypeError: ... 'NoneType' ...` | Something gave back nothing. Usually a function that is missing its `return` line |
| `TypeError: unsupported operand` | Arithmetic on text, or on a whole table where you wanted one column |
| `IndexError` | You asked for item 5 of something with fewer than 6 things in it |

**If several cells stop making sense at once**, the cause is almost always that
they were run out of order. Choose **Kernel** then **Restart Kernel and Run All
Cells**, and work down from the top again.

Try things. Running something wrong and reading the error is how this is
learned, and you cannot break anything.

---
---

## **1. The shots we will work with**

`shots.csv` holds 55,448 tennis shots recorded by SwingVision, a system that
tracks the ball with a camera. Each row is one shot. The columns are:

| Column | What it holds |
|---|---|
| `player` | Which of the two players hit the shot, `host` or `guest` |
| `shot_type` | The role of the shot in the point, for example `first_serve` or `in_play` |
| `hit_type` | How the shot was played, for example `ground_stroke` or `volley` |
| `hit_court_side` | Which half of the court the player was standing in, `near` or `far` |
| `bounce_court_side` | Which half of the court the ball landed in, `near` or `far` |
| `bounce_location_long` | How deep the ball landed, as a category |
| `hit_velocity_x` | Speed sideways at the moment of the hit, in metres per second |
| `hit_velocity_y` | Speed up the court at the moment of the hit, in metres per second |
| `hit_velocity_z` | Speed upwards at the moment of the hit, in metres per second |
| `bounce_y` | How far up the court the ball landed, in metres from the near baseline |

`near` means the half of the court closest to the camera. The court is 23.774
metres long: the near baseline is at 0, the net at 11.887, and the far baseline
at 23.774.

`bounce_y` is measured on that scale, but it is not limited to it. A ball landing
beyond the far baseline records a value above 23.774, and one landing behind the
near baseline records a negative value. In this table `bounce_y` runs from
-3.334 to 26.199. Recorded data also holds occasional tracking errors, which
Section 1 deals with.

In [ ]:
shots = Table().read_table("shots.csv")
print("rows:", shots.num_rows)
shots.show(5)

---

**Question 1.1.** Before filtering anything, find out what values
`bounce_location_long` actually takes.

Assign `bounce_locations` to a two-column table: the first column should hold
each different value that appears in `bounce_location_long`, and the second
should hold how many times each one appears.

`tbl.group('some_column')` does exactly this. It adds up how many rows share
each value and puts the totals in a new column, which it names `count`.
__(3 Points)__

In [ ]:
bounce_locations = shots.group(...)
bounce_locations

In [ ]:
grader.check("q1_1")

So there are three depths: `service_box`, `no_mans_land` and `out`.

These are not opinions about the shot. They are the court's own lines. For a
ball landing in the far half:

* `service_box`: between the net and the far service line, `bounce_y` up to 18.288
* `no_mans_land`: between the service line and the far baseline, 18.289 to 23.774
* `out`: past the baseline, so beyond 23.774

<img src="court.png" alt="a tennis court from above, with the three depths marked" width="620"/>

*The figure above is a tennis court. The far half of the court, seen from above. The near player hits from the left.
Widths are drawn to scale, but `shots.csv` records only how far up the court a
ball landed, not how far across.*

Those two names are the real regions of a tennis court, not labels invented for
this lab. The gap from the net to the far service line works out at 6.401 metres
in this file, and a real service box is 6.40 deep.

---

**Question 1.2.** There is a lot in this table that we do not need, so narrow it
down to the shots we care about:

* shots played as a `ground_stroke` (the `hit_type` column)
* shots hit by the player nearest the camera (the `hit_court_side` column)
* shots that landed in the other half of the court (the `bounce_court_side` column)
* shots that were **not** out (the `bounce_location_long` column)

Then keep only these five columns: `hit_velocity_x`, `hit_velocity_y`,
`hit_velocity_z`, `bounce_location_long` and `bounce_y`. The order you list them
in does not matter. All three velocity columns stay for now; Section 2 works out
which of them are worth using.

Call the result `ground_strokes`.

The first filter is written for you. The `are.` functions are called
**predicates**: each one tests a value and answers yes or no. Any predicate can
be reversed by putting `not_` in front of its name, so `are.not_equal_to('x')`
keeps every row whose value is not `'x'`. __(6 Points)__

In [ ]:
ground_strokes = (shots.where('hit_type', are.equal_to('ground_stroke'))
                       .where(..., are.equal_to('near'))
                       .where(..., are.equal_to('far'))
                       .where(..., are.not_equal_to(...))
                       .select(...))
ground_strokes.show(5)

In [ ]:
grader.check("q1_2")

That fourth filter looks like tidying up. It is not. Run the cell below and
read what is left.

In [ ]:
# Just run this cell.
for depth in ['service_box', 'no_mans_land']:
    landed = ground_strokes.where('bounce_location_long', depth).column('bounce_y')
    print(depth, "-> bounce_y from", round(min(landed), 3), "to", round(max(landed), 3))

---

**Question 1.3.** Which of these best describes what dropping the `out` shots
did to the data? Set `q1_3_answer` to `1`, `2` or `3`.

1. It removed the deepest shots, so no shot left in the table landed further up
   the court than 18.288 metres.
2. It removed the deepest shots, so no shot left in the table landed further up
   the court than 23.774 metres.
3. It removed shots at both ends, so no shot left in the table landed closer than
   18.289 metres or further than 23.774 metres.

__(3 Points)__

In [ ]:
q1_3_answer = ...

In [ ]:
grader.check("q1_3")

Nearest neighbours compares a new shot against every shot it already has, so a
smaller table keeps this lab quick. The cell below takes a random 1,000 of the
ground strokes. `np.random.seed(8)` makes the choice the same every time it is
run, so everyone gets the same numbers.

It then drops four shots whose recorded upward speed is over 15 metres per
second. Straight up at 15 m/s would send the ball roughly 11 metres into the
air off a ground stroke, which does not happen; these are tracking errors. The
cell prints how many are removed so nothing disappears quietly.

In [ ]:
np.random.seed(8)  # do not change this line
sample_1000 = ground_strokes.sample(with_replacement=False).take(np.arange(1000))

coords = sample_1000.where('hit_velocity_z', are.below(15))
print("kept", coords.num_rows, "of", sample_1000.num_rows, "shots")
coords.show(3)

---
---

## **2. Measuring how alike two shots are**

We want to predict where a shot will land from how it left the racket. Two of
the three velocity columns say something about depth:

* `hit_velocity_y`: speed up the court. Faster forward, deeper landing.
* `hit_velocity_z`: speed upwards. More lift, longer flight, deeper landing.

`hit_velocity_x` is sideways speed. It moves the ball left or right, not up the
court, so we leave it out. It stays in the table, and you should not use it.

In [ ]:
# Just run this cell.
far = coords.where('bounce_location_long', 'no_mans_land')
near = coords.where('bounce_location_long', 'service_box')
plt.scatter(far.column('hit_velocity_y'), far.column('hit_velocity_z'),
            s=5, alpha=0.4, color='blue', label='no_mans_land')
plt.scatter(near.column('hit_velocity_y'), near.column('hit_velocity_z'),
            s=5, alpha=0.4, color='orange', label='service_box')
plt.xlabel("Hit velocity, y (m/s)")
plt.ylabel("Hit velocity, z (m/s)")
plt.legend();

The two groups sit in different parts of the picture, but they overlap a great
deal, and there is no line you could draw that would separate them cleanly. Be
careful with pictures like this: whichever group is drawn second covers the
other one, so the overlap always looks smaller than it is.

That overlap is not a reason to give up. Nearest neighbours only needs shots
that are close together to *tend* to land in the same place, which is a much
weaker requirement than a clean boundary. Rather than judging from the picture,
we will measure how often the method is right, at the end of the lab.

---

### **Keeping some data back**

If we let the method see every shot and then ask how well it does on those same
shots, we learn nothing: it can be right simply by remembering. So we split the
data in two. The **training set** is what the method is allowed to look at. The
**test set** is held back, and we only use it at the end to see how often the
method gets an answer right that it could not have memorised.

The `datascience` library has `tbl.split`, which does this in one step. Do not
use it here. Doing it by hand is three lines, and you can see exactly which
rows went where, which matters when a result later looks wrong.

---

**Question 2.1.** Split `coords` into a test set and a training set. Shuffle
first, then put the first 200 shuffled shots in `test_coords` and every
remaining shot in `training_coords`.

`np.arange(a, b)` counts from `a` up to but **not including** `b`, so
`np.arange(0, 3)` gives 0, 1, 2. __(6 Points)__

In [ ]:
np.random.seed(8)  # do not change this line

num_shots = coords.num_rows
shuffled_coords = coords.sample(num_shots, with_replacement=False)

test_coords = shuffled_coords.take(np.arange(...))
training_coords = shuffled_coords.take(np.arange(..., ...))
print(test_coords.num_rows, "test shots and", training_coords.num_rows, "training shots")

In [ ]:
grader.check("q2_1")

---

### **`print` and `return`**

These two are easy to confuse, and the difference matters for the next question.

`print` puts a value on the screen for you to look at. `return` hands the value
back to whatever called the function, so it can be used in more arithmetic.

A function with no `return` line hands back nothing at all. Python calls that
nothing `None`, and arithmetic on `None` fails with
`TypeError: unsupported operand type(s) ... 'NoneType'`. If you ever see
`NoneType` in an error, look for a missing `return`.

In [ ]:
# Just run this cell.
def gap(a, b):
    # Give back the difference between two numbers.
    return a - b

print("gap(10, 4) is", gap(10, 4))
print("and it can be used again:", gap(10, 4) ** 2)

### **One distance, by hand**

Here is the distance between two particular shots, with every step visible. Run
it and read it. `tbl.row(0)` picks out the first row, `row.item('col')` reads one
value from it, and `make_array` gathers those values into an array.

Subtracting one array from another subtracts matching positions, so `gaps` holds
both differences at once.

In [ ]:
# Just run this cell.
def features_of(row):
    # The two measurements we compare shots on, gathered into an array.
    return make_array(row.item('hit_velocity_y'), row.item('hit_velocity_z'))

shot_a = features_of(coords.row(0))
shot_b = features_of(coords.row(1))
gaps = shot_a - shot_b

print("shot a: ", np.round(shot_a, 3))
print("shot b: ", np.round(shot_b, 3))
print("gaps:   ", np.round(gaps, 3))
print("squared:", np.round(gaps ** 2, 3))
print("distance:", round(np.sqrt(sum(gaps ** 2)), 3))

Each gap squared, added together, then square rooted. That is Pythagoras, and it
is the whole idea: the distance between two shots is the straight-line distance
between the two points that represent them in the picture above.

The next question asks for exactly this, written as a function that takes two
arrays. Written that way it does not care how many measurements the arrays hold,
which is what Project 2 needs.

---

**Question 2.2.** Complete `distance`, which takes two arrays of measurements
and gives back the straight-line distance between them.

For two shots $a$ and $b$ measured on $y$ and $z$:

$$\text{distance} = \sqrt{(y_a - y_b)^2 + (z_a - z_b)^2}$$

`features_of` above builds those arrays, and it leaves out `hit_velocity_x`
because sideways speed says nothing about depth. Use array arithmetic rather
than reading values out one at a time, and your function will work for arrays of
any length. `sum(array)` adds up an array. __(6 Points)__

In [ ]:
def distance(features1, features2):
    gaps = ...
    squared = ...
    return ...

In [ ]:
grader.check("q2_2")

---
---

## 3. **Classifying a shot**

Now use the distance. To classify a shot we have never seen: measure how far it
is from every training shot, find the closest one, and guess that our shot lands
where that one did. That is nearest neighbours with $k = 1$.

The function below does the measuring. It walks through the training table one
row at a time, turns each row into an array with `features_of`, calls **your**
`distance` function, collects the answers with `np.append`, and hands back a copy
of the table with a `distance` column added.

In [ ]:
# Just run this cell.
def distances_from(test_row, training_table):
    # Copy training_table, adding the distance from test_row to every row.
    gaps = make_array()
    for training_row in training_table.rows:
        gaps = np.append(gaps, distance(features_of(test_row), features_of(training_row)))
    return training_table.with_columns('distance', gaps)

distances_from(test_coords.row(0), training_coords).sort('distance').show(3)

`tbl.sort(column_name)` reorders the rows so the smallest value comes first. Sort
by the column `distances_from` just added, and the closest training shot is row
0 of the result. What you want from that row is where its ball landed.

---

**Question 3.1.** Complete `closest_label`. It takes one shot and a table of
training shots, and gives back the `bounce_location_long` of the training shot
closest to it. __(6 Points)__

In [ ]:
def closest_label(test_row, training_table):
    with_distances = distances_from(test_row, training_table)
    sorted_table = with_distances.sort(...)
    return sorted_table.column(...).item(0)

In [ ]:
grader.check("q3_1")

---

### How often is it right?

Now use the test set, which nothing has looked at so far. The cell below
classifies the first 50 test shots and compares each answer with what actually
happened. Fifty rather than all 200, because each one is measured against 796
training shots, and doing that 200 times would be slow.

Compare the result against the last line: what you would score by ignoring the
velocities completely and always saying `no_mans_land`, because that is the more
common answer. A method is only worth having if it beats that.

In [ ]:
# Just run this cell. It takes a few seconds.
checked = test_coords.take(np.arange(50))
right = 0
for shot in checked.rows:
    if closest_label(shot, training_coords) == shot.item('bounce_location_long'):
        right = right + 1

common = training_coords.group('bounce_location_long').sort('count', descending=True)
always = np.count_nonzero(checked.column('bounce_location_long') == common.column(0).item(0))
print("nearest neighbour correct:", right, "out of", checked.num_rows)
print("always guessing", common.column(0).item(0) + ":", always, "out of", checked.num_rows)

---

### One neighbour, or more than one?

Beating that always-guessing score is only the first step, but the closest training shot is only one shot,
and one shot can be unlucky. Shot 13 of the test set is one the method above gets
wrong.

In [ ]:
# Just run this cell.
awkward = test_coords.row(13)
nearest_three = distances_from(awkward, training_coords).sort('distance').take(np.arange(3))
nearest_three.select('bounce_location_long', 'distance').show()

vote = nearest_three.group('bounce_location_long').sort('count', descending=True)
print("it actually landed in:  ", awkward.item('bounce_location_long'))
print("the single nearest said:", closest_label(awkward, training_coords))
print("the three nearest say:  ", vote.column(0).item(0))

The closest shot was a `service_box` shot, but the next two both landed in
`no_mans_land`, and two out of three is the right answer. Counting several
neighbours and taking the more common label is what the **k** in nearest
neighbours means: $k = 1$ everywhere above, $k = 3$ here. Across the same fifty
test shots, $k = 3$ gets 41 right where $k = 1$ gets 38.

The three steps are ones you have already used: sort by distance, take the k
nearest, then `group` the labels and sort by the `count` column it creates.
Project 2 asks you to write that vote yourself, as a function called
`most_common`, and runs its classifier at $k = 11$.

Better than always guessing, and still often wrong. That is what the
overlap in the scatter plot looks like when you put a number on it, and it is
worth remembering when a classifier is described as accurate: accurate compared
with what?

One thing to be careful about: the two velocity columns are both measured in
metres per second, so adding their squared gaps is fair. When the columns are
measured in different units, the one with the larger numbers quietly dominates
the distance. That comes up again in Project 2.

---

## **You're done!**

**Important submission information:**
- **Run all the tests** and verify that they all pass
- **Save** from the **File** menu
- **Run the final cell to generate the grader.check_all()**
- **Click the download button to download the .ipynb file (see the figure below)**
- **Then, go to Canvas and submit the .ipynb file to Lab 7: Nearest Neighbours**

**It is your responsibility to make sure your work is saved before running the last cell.**


<img src="download_ipynb.png" alt="download ipynb" width="800"/>

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## **Submission**

Make sure you have run all cells in your notebook in order before clicking the download button to download the .ipynb file. **Please save your work before downloading!**